In [ ]:
!pip install lightgbm shap

In [ ]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
import lightgbm as lgb
import joblib
import shap

In [ ]:
# --- 設定値 ---
# Kaggle「Students Performance in Exams」のCSVをここに配置してください
# https://www.kaggle.com/datasets/spscientist/students-performance-in-exams
CSV_PATH = "data/StudentsPerformance.csv"
MODEL_OUTPUT_DIR = "app/models/math_predictor/v1"

In [ ]:
# --- ローカルCSVからデータを読み込む ---
df = pd.read_csv(CSV_PATH)

# 列名のスペースをアンダースコアに変換（race/ethnicityはスラッシュを維持）
df = df.rename(columns={
    "parental level of education": "parental_level_of_education",
    "test preparation course": "test_preparation_course",
    "math score": "math_score",
    "reading score": "reading_score",
    "writing score": "writing_score",
})

In [ ]:
# --- 特徴量と目的変数を定義 ---
X = df.drop('math_score', axis=1)
y = df['math_score']

In [ ]:
# --- データ分割 ---
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
# --- 前処理（One-Hot エンコーディング） ---
categorical_cols = X_train.select_dtypes(include=['object']).columns
X_train_encoded = pd.get_dummies(X_train, columns=categorical_cols, drop_first=True)
X_test_encoded = pd.get_dummies(X_test, columns=categorical_cols, drop_first=True)
final_columns = X_train_encoded.columns

In [ ]:
final_columns
results = {}
models = {}

In [ ]:
# --- モデル学習と評価 ---
lr_model = LinearRegression()
lr_model.fit(X_train_encoded, y_train)
lr_pred = lr_model.predict(X_test_encoded)
results["LinearRegression"] = r2_score(y_test, lr_pred)
models["LinearRegression"] = lr_model

lgb_model = lgb.LGBMRegressor(random_state=42)
lgb_model.fit(X_train_encoded, y_train)
lgb_pred = lgb_model.predict(X_test_encoded)
results["LightGBM"] = r2_score(y_test, lgb_pred)
models["LightGBM"] = lgb_model

print("モデル比較結果:", results)

In [ ]:
# --- SHAP値の計算 (LightGBM用) ---
explainer = shap.TreeExplainer(lgb_model)
shap_values = explainer(X_test_encoded)
shap.summary_plot(shap_values, X_test_encoded, show=False)  # 可視化

In [ ]:
# --- モデルと列情報をローカルに保存（appが読み込むディレクトリ） ---
os.makedirs(MODEL_OUTPUT_DIR, exist_ok=True)

for name, model in models.items():
    joblib.dump(model, os.path.join(MODEL_OUTPUT_DIR, f"{name}.pkl"))

joblib.dump(list(final_columns), os.path.join(MODEL_OUTPUT_DIR, "feature_list.pkl"))

print(f"モデルと特徴量リストを {MODEL_OUTPUT_DIR} に保存しました")